# Chapter 17: Confidence Calibration via Phase Encoding

**Learning objectives:**
- Understand the S^1 phase encoding of confidence and accuracy
- Measure ECE (Expected Calibration Error) and MCE (Maximum Calibration Error)
- Build and interpret reliability diagrams
- Generate calibration probes via `CalibrationGenerator`

---
## Preamble

Chapter 16 detected hallucination by measuring geodesic distance on the GMS manifold. This chapter addresses a complementary concern: **confidence calibration** — does the agent's self-reported confidence match its actual accuracy?

GMSH encodes both quantities as phases on the unit circle $S^1$:

- **Confidence** $c \in [0, 1]$ maps to angle $\theta_c = c \cdot \pi$ (0 = no confidence, $\pi$ = total confidence)
- **Accuracy** maps to $\theta_a = \pi$ (correct) or $\theta_a = 0$ (incorrect)
- **Calibration gap** = $|\theta_c - \theta_a| / \pi \in [0, 1]$

A perfectly calibrated agent has a gap of 0 when confident and correct, or when uncertain and wrong.

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
import math
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
    enable_tracing=False,
)
bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"Store: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## Phase Encoding on $S^1$

The calibration framework maps confidence and correctness to angles on the unit circle. Three helper functions drive the encoding:

| Function | Input | Output | Range |
|---|---|---|---|
| `_confidence_to_phase(c)` | $c \in [0,1]$ | $\theta = c \cdot \pi$ | $[0, \pi]$ |
| `_accuracy_to_phase(correct)` | bool | $\pi$ if True, $0$ if False | $\{0, \pi\}$ |
| `_angular_distance(a, b)` | two phases | $|a - b| / \pi$ | $[0, 1]$ |

In [ ]:
from knowlytix.harness.testing.calibration import (
    CalibrationTester, CalibrationGenerator,
    _confidence_to_phase, _accuracy_to_phase, _angular_distance,
)

# Confidence -> Phase mapping
print("=== Confidence to Phase ===")
print(f"{'Confidence':<12s} {'Phase (rad)':>12s} {'Phase (deg)':>12s}")
print("-" * 38)
for conf in [0.0, 0.25, 0.5, 0.75, 1.0]:
    phase = _confidence_to_phase(conf)
    print(f"{conf:<12.2f} {phase:>12.4f} {math.degrees(phase):>12.1f}")

# Accuracy -> Phase mapping
print("\n=== Accuracy to Phase ===")
for correct in [True, False]:
    phase = _accuracy_to_phase(correct)
    print(f"  correct={str(correct):<6s} -> phase={phase:.4f} ({math.degrees(phase):.0f} deg)")

In [ ]:
# Angular distance = calibration gap
print("=== Calibration Gap (angular distance) ===")
print(f"{'Confidence':<12s} {'Correct':<8s} {'Gap':>6s}  Interpretation")
print("-" * 60)
cases = [
    (0.9, True,  "Well-calibrated: high confidence, correct"),
    (0.9, False, "Overconfident: high confidence, wrong"),
    (0.1, True,  "Underconfident: low confidence, correct"),
    (0.1, False, "Well-calibrated: low confidence, wrong"),
    (0.5, True,  "Uncertain: mid confidence, correct"),
    (0.5, False, "Uncertain: mid confidence, wrong"),
]
for conf, correct, interp in cases:
    gap = _angular_distance(
        _confidence_to_phase(conf), _accuracy_to_phase(correct)
    )
    print(f"{conf:<12.1f} {str(correct):<8s} {gap:>6.4f}  {interp}")

---

## CalibrationTester and CalibrationReport

The `CalibrationTester` collects `CalibrationPoint` objects via `add()` and produces a `CalibrationReport` via `report()`. The report includes:

| Field | Type | Description |
|---|---|---|
| `ece` | float | Expected Calibration Error: weighted average of per-bin gaps |
| `mce` | float | Maximum Calibration Error: worst-case bin gap |
| `mean_confidence` | float | Mean claimed confidence across all points |
| `mean_accuracy` | float | Actual accuracy rate |
| `overconfidence_ratio` | float | Fraction of bins where confidence > accuracy |
| `bins` | list[CalibrationBin] | Reliability diagram bins |

In [ ]:
# Basic CalibrationTester with 5 hand-crafted points
tester = CalibrationTester(n_bins=10)

tester.add(claimed_confidence=0.95, actual_correct=True, question_id="q1")
tester.add(claimed_confidence=0.90, actual_correct=True, question_id="q2")
tester.add(claimed_confidence=0.85, actual_correct=False, question_id="q3")  # overconfident
tester.add(claimed_confidence=0.30, actual_correct=False, question_id="q4")
tester.add(claimed_confidence=0.60, actual_correct=True, question_id="q5")

report = tester.report()
print("=== CalibrationReport (5 points) ===")
print(f"  ECE (Expected Calibration Error): {report.ece:.4f}")
print(f"  MCE (Maximum Calibration Error):  {report.mce:.4f}")
print(f"  Mean confidence: {report.mean_confidence:.4f}")
print(f"  Mean accuracy:   {report.mean_accuracy:.4f}")
print(f"  Overconfidence ratio: {report.overconfidence_ratio:.0%}")

In [ ]:
# Reliability diagram bins
print("=== Reliability Diagram ===")
print(f"{'Bin':<16s} {'N':>4s} {'Confidence':>11s} {'Accuracy':>9s} {'Gap':>6s}")
print("-" * 50)
for b in report.bins:
    if b.n_samples > 0:
        print(f"[{b.bin_lower:.2f}-{b.bin_upper:.2f})  {b.n_samples:>4d} "
              f"{b.mean_confidence:>11.4f} {b.mean_accuracy:>9.4f} {b.gap:>6.3f}")

In [ ]:
# Per-point detail
print("=== Per-Point Calibration ===")
print(f"{'QID':<5s} {'Conf':>5s} {'Correct':<8s} {'Conf Phase':>11s} "
      f"{'Acc Phase':>10s} {'Gap':>6s}")
print("-" * 50)
for p in tester.points:
    print(f"{p.question_id:<5s} {p.claimed_confidence:>5.2f} "
          f"{str(p.actual_correct):<8s} {p.confidence_phase:>11.4f} "
          f"{p.accuracy_phase:>10.4f} {p.calibration_gap:>6.4f}")

---

## Calibration Profiles

We simulate three agent profiles to illustrate the calibration spectrum:

1. **Well-calibrated**: confidence matches accuracy (low ECE)
2. **Overconfident**: consistently high confidence regardless of correctness (high ECE)
3. **Underconfident**: consistently low confidence even when correct (high ECE)

In [ ]:
import random
random.seed(42)
np.random.seed(42)

def simulate_profile(name, conf_fn, acc_fn, n=100):
    """Simulate a calibration profile."""
    t = CalibrationTester(n_bins=10)
    for i in range(n):
        correct = acc_fn(i)
        conf = conf_fn(i, correct)
        t.add(claimed_confidence=conf, actual_correct=correct,
              question_id=f"{name}_{i}")
    return t.report()

# 1. Well-calibrated: confidence ~ accuracy
r_well = simulate_profile(
    "well_calibrated",
    conf_fn=lambda i, c: min(1.0, max(0.0, (0.85 if c else 0.15) + np.random.normal(0, 0.05))),
    acc_fn=lambda i: random.random() < 0.7,
)

# 2. Overconfident: always high confidence
r_over = simulate_profile(
    "overconfident",
    conf_fn=lambda i, c: min(1.0, max(0.0, 0.90 + np.random.normal(0, 0.03))),
    acc_fn=lambda i: random.random() < 0.5,
)

# 3. Underconfident: always low confidence
r_under = simulate_profile(
    "underconfident",
    conf_fn=lambda i, c: min(1.0, max(0.0, 0.20 + np.random.normal(0, 0.05))),
    acc_fn=lambda i: random.random() < 0.7,
)

print(f"{'Profile':<18s} {'ECE':>8s} {'MCE':>8s} {'Mean Conf':>10s} "
      f"{'Mean Acc':>9s} {'Overconf':>9s}")
print("-" * 65)
for name, r in [("Well-calibrated", r_well), ("Overconfident", r_over),
                ("Underconfident", r_under)]:
    print(f"{name:<18s} {r.ece:>8.4f} {r.mce:>8.4f} {r.mean_confidence:>10.4f} "
          f"{r.mean_accuracy:>9.4f} {r.overconfidence_ratio:>9.0%}")

In [ ]:
# Reliability diagram for each profile
for name, r in [("Well-calibrated", r_well), ("Overconfident", r_over),
                ("Underconfident", r_under)]:
    print(f"\n=== {name} Reliability Diagram ===")
    print(f"{'Bin':<16s} {'N':>4s} {'Conf':>6s} {'Acc':>6s} {'Gap':>6s}")
    print("-" * 42)
    for b in r.bins:
        if b.n_samples > 0:
            marker = "***" if b.gap > 0.3 else ""
            print(f"[{b.bin_lower:.2f}-{b.bin_upper:.2f})  {b.n_samples:>4d} "
                  f"{b.mean_confidence:>6.2f} {b.mean_accuracy:>6.2f} "
                  f"{b.gap:>6.3f} {marker}")

---

## Lab 17.1 -- Measure Agent Confidence Calibration

In this lab we generate calibration probes, simulate agent responses with varying confidence profiles and measure ECE/MCE.

In [ ]:
# Lab 17.1a -- Generate calibration probes
gen = CalibrationGenerator()
probes = gen.generate(store)

types = Counter(p.metadata["test_type"] for p in probes)
print(f"Generated {len(probes)} calibration probes")
print(f"By type: {dict(sorted(types.items()))}")

print(f"\n{'QID':<28s} {'Type':<28s} {'Ground Truth'}")
print("-" * 80)
for p in probes[:5]:
    gt = str(p.ground_truth)[:25]
    print(f"{p.qid:<28s} {p.metadata['test_type']:<28s} {gt}")
print("...")
for p in probes[-3:]:
    gt = str(p.ground_truth)[:25]
    print(f"{p.qid:<28s} {p.metadata['test_type']:<28s} {gt}")

In [ ]:
# Lab 17.1b -- Simulate agent on calibration probes
# The mock agent can't express confidence, so we simulate three profiles
# using the probe's expected_high_confidence metadata

np.random.seed(42)
random.seed(42)

def simulate_agent_on_probes(probes, profile_name, conf_fn, correct_rate):
    """Simulate an agent answering calibration probes."""
    tester = CalibrationTester(n_bins=10)
    for p in probes:
        correct = random.random() < correct_rate
        expected_high = p.metadata.get("expected_high_confidence", True)
        conf = conf_fn(correct, expected_high)
        tester.add(claimed_confidence=conf, actual_correct=correct,
                   question_id=p.qid)
    return tester.report()

# Well-calibrated agent
r_cal = simulate_agent_on_probes(
    probes, "calibrated",
    conf_fn=lambda c, eh: min(1.0, max(0.0, (0.85 if c else 0.2) + np.random.normal(0, 0.05))),
    correct_rate=0.7,
)

# Overconfident agent
r_oc = simulate_agent_on_probes(
    probes, "overconfident",
    conf_fn=lambda c, eh: min(1.0, max(0.0, 0.92 + np.random.normal(0, 0.03))),
    correct_rate=0.5,
)

# Underconfident agent
r_uc = simulate_agent_on_probes(
    probes, "underconfident",
    conf_fn=lambda c, eh: min(1.0, max(0.0, 0.15 + np.random.normal(0, 0.05))),
    correct_rate=0.7,
)

print(f"{'Profile':<18s} {'ECE':>8s} {'MCE':>8s} {'Mean Conf':>10s} "
      f"{'Mean Acc':>9s} {'Overconf':>9s}")
print("-" * 65)
for name, r in [("Well-calibrated", r_cal), ("Overconfident", r_oc),
                ("Underconfident", r_uc)]:
    print(f"{name:<18s} {r.ece:>8.4f} {r.mce:>8.4f} {r.mean_confidence:>10.4f} "
          f"{r.mean_accuracy:>9.4f} {r.overconfidence_ratio:>9.0%}")

---

## Lab 17.2 -- Full DOE Benchmark with Calibration

We run the 640-question DOE benchmark and simulate calibration measurements to compute ECE and MCE at scale.

In [ ]:
from importlib.resources import files
# Lab 17.2a -- Run 640 questions with simulated calibration
from knowlytix.harness.testing.adapters import AgentAdapter
from knowlytix.harness.fixtures import FullMockHarness

adapter = AgentAdapter(
    harness=FullMockHarness(recovery_mode="fallback"),
)
context = open(str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md"))).read()

doe_questions = bench.generate_questions()

# Simulate a realistic agent: 70% accuracy, confidence correlated with correctness
np.random.seed(42)
random.seed(42)

tester_e2e = CalibrationTester(n_bins=10)
for i, dq in enumerate(doe_questions):
    trace = adapter.evaluate(
        question=dq.question.natural_language,
        context=context,
    )
    # Check correctness: does output contain ground truth?
    gt = str(dq.question.ground_truth)
    correct = gt.lower() in trace.raw_output.lower() if gt else False
    # Simulate confidence: correlated with correctness + noise
    base_conf = 0.80 if correct else 0.35
    conf = min(1.0, max(0.0, base_conf + np.random.normal(0, 0.12)))
    tester_e2e.add(claimed_confidence=conf, actual_correct=correct,
                   question_id=f"doe_{i:04d}")

report_e2e = tester_e2e.report()
print(f"=== E2E Calibration Report ({report_e2e.n_points} questions) ===")
print(f"  ECE: {report_e2e.ece:.4f}")
print(f"  MCE: {report_e2e.mce:.4f}")
print(f"  Mean confidence: {report_e2e.mean_confidence:.4f}")
print(f"  Mean accuracy:   {report_e2e.mean_accuracy:.4f}")
print(f"  Overconfidence ratio: {report_e2e.overconfidence_ratio:.0%}")

In [ ]:
# Lab 17.2b -- Reliability diagram at scale
print("=== Reliability Diagram (640 questions) ===")
print(f"{'Bin':<16s} {'N':>5s} {'Conf':>6s} {'Acc':>6s} {'Gap':>6s}")
print("-" * 43)
for b in report_e2e.bins:
    if b.n_samples > 0:
        marker = "  ***" if b.gap > 0.3 else ""
        print(f"[{b.bin_lower:.2f}-{b.bin_upper:.2f})  {b.n_samples:>5d} "
              f"{b.mean_confidence:>6.2f} {b.mean_accuracy:>6.2f} "
              f"{b.gap:>6.3f}{marker}")

In [ ]:
# Lab 17.2c -- Compare n_bins effect on ECE/MCE
# Re-use the same calibration points with different binning
points = tester_e2e.points

print("=== Bin Count Sensitivity ===")
print(f"{'n_bins':>7s} {'ECE':>8s} {'MCE':>8s} {'Non-empty bins':>15s}")
print("-" * 42)
for n_bins in [5, 10, 15, 20]:
    t = CalibrationTester(n_bins=n_bins)
    for p in points:
        t.add(claimed_confidence=p.claimed_confidence,
              actual_correct=p.actual_correct,
              question_id=p.question_id)
    r = t.report()
    nonempty = sum(1 for b in r.bins if b.n_samples > 0)
    print(f"{n_bins:>7d} {r.ece:>8.4f} {r.mce:>8.4f} {nonempty:>15d}")

In [ ]:
# Per-point calibration gap distribution
gaps = [p.calibration_gap for p in points]

print("=== Calibration Gap Distribution (640 points) ===")
print(f"  Mean gap:   {np.mean(gaps):.4f}")
print(f"  Std gap:    {np.std(gaps):.4f}")
print(f"  Median gap: {np.median(gaps):.4f}")
print(f"  Min gap:    {np.min(gaps):.4f}")
print(f"  Max gap:    {np.max(gaps):.4f}")

# Distribution by bucket
print("\n=== Gap Buckets ===")
buckets = [(0, 0.1, "excellent"), (0.1, 0.2, "good"),
           (0.2, 0.4, "moderate"), (0.4, 0.6, "poor"),
           (0.6, 1.01, "severe")]
for lo, hi, label in buckets:
    n = sum(1 for g in gaps if lo <= g < hi)
    print(f"  [{lo:.1f}-{hi:.1f}) {label:<10s}: {n:>4d} ({100*n/len(gaps):.1f}%)")

---

## Exercises

**Exercise 17.1** -- Create a `CalibrationTester` with `n_bins=5` and add 50 points where the agent always claims confidence 0.5. Compute the ECE and MCE. What does constant-confidence tell you about the agent's self-awareness?

**Exercise 17.2** -- Use the `CalibrationGenerator` to generate probes. For each answerable probe, simulate an agent that is correct 80% of the time with confidence `0.8 + noise(0, 0.1)`. For unanswerable probes, simulate 50% accuracy with the same confidence. Compare the ECE for answerable vs unanswerable subsets.

**Exercise 17.3** -- Run the 640-question benchmark three times with simulated accuracy rates of 50%, 70% and 90%, keeping the confidence distribution fixed at `0.7 + noise(0, 0.1)`. Plot ECE vs accuracy rate. At what accuracy rate is the agent best calibrated for this confidence distribution?

---

## Summary

This chapter introduced **confidence calibration via phase encoding** on $S^1$:

1. **Phase encoding** maps confidence $c \in [0,1]$ to angle $\theta_c = c \cdot \pi$ and correctness to $\theta_a \in \{0, \pi\}$. The calibration gap is the normalized angular distance.

2. **ECE** (Expected Calibration Error) is the weighted average of per-bin accuracy-confidence gaps. A well-calibrated agent has ECE close to 0.

3. **MCE** (Maximum Calibration Error) is the worst-case bin gap — it reveals the confidence region where the agent is most miscalibrated.

4. **Reliability diagrams** visualize calibration by plotting mean confidence vs mean accuracy per bin. Bins above the diagonal indicate overconfidence; below indicates underconfidence.

5. **CalibrationGenerator** produces answerable and unanswerable probes. A well-calibrated agent should express high confidence on answerable questions and low confidence on unanswerable ones.

In the next chapter we turn to **stateful and multi-turn testing** — verifying that agents maintain memory across conversation turns and degrade gracefully under sustained interaction.